# Sector Financiero: créditos y depósitos (SBS)
Procesa los seis cuadros mensuales de la SBS (uno por tipo de entidad) y genera los outputs del mes.
Los archivos se reconocen por su **título y cabecera**, no por el nombre: sirven tanto los cuadros
"por Oficina y Empresa" (BANCA_MULTIPLE-*.XLS, etc.) como los descargados en crudo (B-2314, B-3254, B-5215,
C-1213, C-2213, C-4203). Se detecta si un .XLS es en realidad .xlsx, si un cuadro viene en millones
o con fecha distinta al resto, y si repite exactamente el mes anterior. Si un tipo viene en dos cuadros (por oficina y
agregado), se usa el de oficina y el agregado sirve de validación (hoja "Validación" en los Excel por departamento).

**Cada mes solo cambian `ANIO` y `MES`** (estructura `Financiero/<AAAA>/<N.MesAAAA>/`). Inputs en `input/`: un archivo por tipo (Banca Múltiple, Empresas Financieras,
Cajas Municipales, Cajas Rurales, Empresas de Crédito, Banco de la Nación y Agrobanco).
Solo la primera vez: colocar también en `input/` el libro histórico `BANCA_MULTIPLE_SBS*.xlsx`
(se copia a la raíz de `Financiero/` como libro acumulado y sus hojas siembran el histórico).

Outputs en `output/`:
- **`BANCA_MULTIPLE_SBS-<MesAA>.xlsx`** (output principal): libro acumulado con la hoja del mes, con los tres bloques
  Loreto, San Martín y Ucayali por oficina y entidad, subtotales con fórmulas. Si algún cuadro SBS viene agregado
  (sin entidad), ese tipo entra por distrito y la hoja lo señala.
- **`REPORTE_<AAAAMM>.txt`**: aperturas y cierres de oficinas respecto al mes anterior, y siempre las advertencias
  sobre los inputs (fechas distintas, unidades, cuadros repetidos, falta de detalle por oficina).
- `Financiero_<Depto>_<AAAAMM>.xlsx`, uno por departamento (Loreto, San Martín, Ucayali): resumen (con advertencias),
  cuadros por tipo, entidad, provincia, distrito, ranking nacional y variaciones; más gráficos `g1`..`g5` por departamento
  en PNG y `panel_oficinas_nacional_<AAAAMM>.csv`.

En la raíz de `Financiero/` se mantienen `BANCA_MULTIPLE_SBS.xlsx` (libro acumulado) y `panel_historico_oficinas.csv`
(todo el país, mes a mes). Reprocesar un mes reemplaza sus datos en ambos.

In [ ]:
# 1. Montar Drive y definir rutas
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os

SECTOR       = 'Financiero'
ANIO         = '2026'                 # <-- carpeta del año
MES          = '8.Agosto2026'         # <-- carpeta del mes (dentro del año)

BASE_SECTOR = f'/content/drive/My Drive/{SECTOR}'          # aquí viven el libro acumulado y el histórico
BASE_DIR    = os.path.join(BASE_SECTOR, ANIO, MES)
INPUT_DIR   = os.path.join(BASE_DIR, 'input')
OUTPUT_DIR  = os.path.join(BASE_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(INPUT_DIR)

print("Directorio actual:", os.getcwd())
print("Contenido de input:")
for nombre in sorted(os.listdir(INPUT_DIR)):
    print(" -", nombre)

In [ ]:
# 2. Librerías
!pip install -q xlrd openpyxl
import re, glob, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.1f}'.format)
MESES_ES = ['enero','febrero','marzo','abril','mayo','junio','julio','agosto','setiembre','octubre','noviembre','diciembre']

In [ ]:
# 3. Lector de los archivos SBS
# Reconoce el cuadro por su título y cabecera (no por el nombre del archivo). Tres diseños:
#   A) "por Oficina y Empresa": Empresa + Departamento/Provincia/Distrito + Código de oficina  (nivel = oficina)
#   B) "por Zona Geográfica":   Departamento/Provincia/Distrito, sin entidad                 (nivel = distrito)
#   C) "Departamento y Distrito": una sola columna jerárquica, sin provincia ni entidad       (nivel = distrito)
# Algunos .XLS son en realidad .xlsx; se detecta por el contenido.

TIPO_POR_TITULO = [            # (texto en el título, tipo)
    ('banca multiple', 'Banca Múltiple'), ('empresa bancaria', 'Banca Múltiple'),
    ('financiera', 'Empresas Financieras'), ('cajas municipales', 'Cajas Municipales'),
    ('cajas rurales', 'Cajas Rurales'), ('empresas de credito', 'Empresas de Crédito'),
    ('banco de la nacion', 'Entidades Estatales'),
]
ORDEN_TIPOS = ['Banca Múltiple', 'Empresas Financieras', 'Cajas Municipales', 'Cajas Rurales',
               'Empresas de Crédito', 'Entidades Estatales']
DEPARTAMENTOS_PERU = ['Amazonas', 'Ancash', 'Apurimac', 'Arequipa', 'Ayacucho', 'Cajamarca', 'Callao', 'Cusco',
                      'Huancavelica', 'Huanuco', 'Ica', 'Junin', 'La Libertad', 'Lambayeque', 'Lima', 'Loreto',
                      'Madre de Dios', 'Moquegua', 'Pasco', 'Piura', 'Puno', 'San Martin', 'Tacna', 'Tumbes', 'Ucayali']

COLS_VALOR = ['dep_vista_mn', 'dep_vista_me', 'dep_vista_total',
              'dep_ahorro_mn', 'dep_ahorro_me', 'dep_ahorro_total',
              'dep_plazo_mn', 'dep_plazo_me', 'dep_plazo_total', 'dep_total',
              'cred_mn', 'cred_me', 'cred_total']
COLS_ID = ['fecha', 'tipo', 'nivel', 'empresa', 'departamento', 'provincia', 'distrito', 'cod_oficina']


def _engine(path):
    with open(path, 'rb') as h:
        return 'openpyxl' if h.read(2) == b'PK' else 'xlrd'


def _norm(s):
    s = '' if pd.isna(s) else str(s).strip()
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    return re.sub(r'[^A-Za-z]', '', s).lower()


def _nombre_columna(top, sub):
    """Traduce el encabezado de dos niveles al nombre interno; None si no es columna de valores."""
    t, s = _norm(top), _norm(sub)
    base = {'depositosalavista': 'dep_vista', 'depositosdeahorro': 'dep_ahorro', 'depositosaplazo': 'dep_plazo',
            'totaldepositos': 'dep_total', 'depositostotales': 'dep_total',
            'creditosdirectos': 'cred', 'totalcreditos': 'cred_total'}.get(t)
    if base is None:
        return None
    if base in ('dep_total', 'cred_total'):
        return base + {'mn': '_mn', 'me': '_me'}.get(s, '')        # el "Total" del total es la columna misma
    return base + {'mn': '_mn', 'me': '_me', 'total': '_total'}[s]


def leer_sbs(path):
    raw = pd.read_excel(path, engine=_engine(path), header=None)
    nombre = os.path.basename(path)

    # título, fecha y tipo
    textos = [str(v) for v in raw.iloc[:6].values.ravel() if isinstance(v, str)]
    titulo = next((t for t in textos if 'cr' in t.lower() and 'dep' in t.lower() or 'oficina' in t.lower()), textos[0])
    tipo = next((tp for k, tp in TIPO_POR_TITULO if k.replace(' ', '') in _norm(titulo)), None)
    assert tipo is not None, f'{nombre}: no se reconoce el tipo de entidad en el título "{titulo}"'
    fecha = next(pd.to_datetime(v) for v in raw.iloc[:6].values.ravel() if isinstance(v, (pd.Timestamp, datetime.datetime)))

    # fila de encabezado y diseño
    c0 = raw[0].astype(str).str.strip().str.lower()
    if (c0 == 'empresa').any():
        diseno, h = 'A', raw.index[c0 == 'empresa'][0]
    elif c0.str.startswith('departamento y distrito').any():
        diseno, h = 'C', raw.index[c0.str.startswith('departamento y distrito')][0]
    elif (c0 == 'departamento').any():
        diseno, h = 'B', raw.index[c0 == 'departamento'][0]
    else:
        raise ValueError(f'{nombre}: diseño de cuadro no reconocido')

    top, sub = raw.iloc[h].ffill(), raw.iloc[h + 1]
    cols = []
    for j, (t, s) in enumerate(zip(top, sub)):
        tn = _norm(t)
        if j == 0 and diseno == 'A':      cols.append('empresa')
        elif j == 0:                      cols.append('departamento')
        elif tn == 'ubicacion':           cols.append(_norm(s))
        elif tn in ('provincia', 'distrito'): cols.append(tn)
        elif tn in ('codigodeoficina', 'codigooficina'): cols.append('cod_oficina')
        else:                             cols.append(_nombre_columna(t, s) or f'_x{j}')
    df = raw.iloc[h + 2:].copy()
    df.columns = cols
    df = df.loc[:, ~df.columns.str.startswith('_x')]

    # fin del cuadro: primer "Total general" / "Total Nacional" (Estatales trae un 2º bloque en millones)
    etiqueta = df.iloc[:, 0].astype(str).str.strip().str.lower()
    fin = df.index[etiqueta.str.startswith(('total general', 'total nacional'))]
    if len(fin):
        df = df.loc[:fin[0] - 1]
    df = df[~df.iloc[:, 0].astype(str).str.strip().str.lower().str.startswith('nota')]

    num = [c for c in df.columns if c.startswith(('dep_', 'cred'))]
    df[num] = df[num].apply(pd.to_numeric, errors='coerce')

    if diseno == 'A':
        df = df[df['cod_oficina'].notna() | df[num].notna().any(axis=1)]
        for c in ['empresa', 'departamento', 'provincia', 'distrito']:
            df[c] = df[c].ffill().astype(str).str.strip()
        df['nivel'] = 'oficina'
    elif diseno == 'B':
        df = df[~df['departamento'].astype(str).str.strip().str.lower().str.startswith('total')]
        for c in ['departamento', 'provincia']:
            df[c] = df[c].ffill()
        df = df[df['distrito'].notna()].copy()
        for c in ['departamento', 'provincia', 'distrito']:
            df[c] = df[c].astype(str).str.strip()
        df['empresa'] = None; df['cod_oficina'] = np.nan; df['nivel'] = 'distrito'
    else:  # C: jerarquía departamento > distrito en una sola columna
        df = df[df['departamento'].notna()].copy()
        nombres = df['departamento'].astype(str).str.strip()
        dep_actual, deptos, es_depto = None, [], []
        for n in nombres:
            if n in DEPARTAMENTOS_PERU and n != dep_actual:
                dep_actual = n; es_depto.append(True)
            else:
                es_depto.append(False)
            deptos.append(dep_actual)
        df['distrito'] = nombres.values; df['departamento'] = deptos
        chk = df.copy(); chk['es_depto'] = es_depto
        sub_ok = np.isclose(chk.loc[chk['es_depto'], 'cred_total'].sum(), chk.loc[~chk['es_depto'], 'cred_total'].sum(), rtol=1e-6)
        if not sub_ok:
            print(f'  AVISO {nombre}: la suma de distritos no cuadra con los subtotales departamentales')
        df = df[~np.array(es_depto)].copy()
        df['provincia'] = None; df['empresa'] = None; df['cod_oficina'] = np.nan; df['nivel'] = 'distrito'

    df[num] = df[num].fillna(0.0)
    for c in COLS_VALOR:
        if c not in df.columns:
            df[c] = 0.0
    df['tipo'] = tipo; df['fecha'] = fecha
    df = df[COLS_ID + COLS_VALOR].reset_index(drop=True)
    print(f"{tipo:22s} diseño {diseno} ({'oficina' if diseno == 'A' else 'distrito'}) | {fecha:%d/%m/%Y} | "
          f"filas={len(df):5d} | créditos={df['cred_total'].sum()/1e3:12,.1f} mill. | depósitos={df['dep_total'].sum()/1e3:12,.1f} mill. | {nombre}")
    return df


import datetime
archivos = sorted(glob.glob(os.path.join(INPUT_DIR, '*.XLS*')) + glob.glob(os.path.join(INPUT_DIR, '*.xls*')))
archivos = [a for a in archivos if 'SBS' not in os.path.basename(a).upper()]        # excluye el libro acumulado
frames = [leer_sbs(f) for f in archivos]

# Si un tipo viene en dos cuadros (por oficina y agregado), se usa el de oficina y el otro sirve para validar
principales, validacion = [], []
for t in ORDEN_TIPOS:
    cand = [f for f in frames if f['tipo'].iloc[0] == t]
    if not cand:
        continue
    cand.sort(key=lambda f: (f['nivel'].iloc[0] != 'oficina', -f['fecha'].iloc[0].value))   # oficina primero, luego más reciente
    principales.append(cand[0])
    for extra in cand[1:]:
        a = cand[0].groupby('departamento')[['cred_total', 'dep_total']].sum()
        e = extra.groupby('departamento')[['cred_total', 'dep_total']].sum()
        comp = a.join(e, lsuffix='_oficina', rsuffix='_agregado', how='outer').fillna(0)
        comp.insert(0, 'tipo', t)
        comp['dif_cred_%'] = 100 * (comp['cred_total_oficina'] / comp['cred_total_agregado'].replace(0, np.nan) - 1)
        comp['dif_dep_%'] = 100 * (comp['dep_total_oficina'] / comp['dep_total_agregado'].replace(0, np.nan) - 1)
        validacion.append(comp.reset_index())
        difs = comp.loc[comp['dif_cred_%'].abs() > 0.5]
        print(f"VALIDACIÓN {t}: cuadro por oficina vs agregado | créditos nacionales "
              f"{a['cred_total'].sum()/1e3:,.1f} vs {e['cred_total'].sum()/1e3:,.1f} mill. | "
              f"departamentos con diferencia > 0,5%: {len(difs)}" + (f" ({', '.join(difs.index)})" if len(difs) else ''))
panel = pd.concat(principales, ignore_index=True)
validacion = pd.concat(validacion, ignore_index=True) if validacion else None
faltan = [t for t in ORDEN_TIPOS if t not in set(panel['tipo'])]
assert not faltan, f'Faltan archivos SBS para: {faltan}'

# fecha de corte: la más frecuente; archivos con otra fecha se marcan y se usan como proxy
ADVERTENCIAS = []
FECHA = panel['fecha'].mode()[0]
for t, f in panel.groupby('tipo', observed=True)['fecha'].first().items():
    if f != FECHA:
        ADVERTENCIAS.append(f'{t}: el archivo SBS está fechado al {f:%d/%m/%Y}, no al {FECHA:%d/%m/%Y}; se usa como proxy del mes.')
panel['fecha'] = FECHA
panel['tipo'] = pd.Categorical(panel['tipo'], ORDEN_TIPOS, ordered=True)
TAG = FECHA.strftime('%Y%m')
NIVEL_OFICINA = panel.groupby('tipo', observed=True)['nivel'].agg(lambda s: (s == 'oficina').all())
TODO_POR_OFICINA = bool(NIVEL_OFICINA.all())
if not TODO_POR_OFICINA:
    ADVERTENCIAS.append('Sin detalle por oficina y entidad para: ' + ', '.join(NIVEL_OFICINA.index[~NIVEL_OFICINA]) +
                        '. No se genera la hoja del libro BANCA_MULTIPLE_SBS ni el reporte de aperturas/cierres; '
                        'para ello se necesitan los cuadros "Depósitos y Créditos por Oficina y Empresa".')
print(f"\nPanel nacional: {len(panel):,} filas | fecha de corte {FECHA:%d/%m/%Y} | miles de soles")

In [ ]:
# 4. Panel histórico del sector (se acumula mes a mes en la raíz de Financiero)
HIST = os.path.join(BASE_SECTOR, 'panel_historico_oficinas.csv')
panel['fuente'] = 'sbs'
if os.path.exists(HIST):
    hist = pd.read_csv(HIST, parse_dates=['fecha'])
    if 'nivel' not in hist.columns: hist['nivel'] = 'oficina'
    if 'fuente' not in hist.columns: hist['fuente'] = 'sbs'
    hist = hist[hist['fecha'] != FECHA]                 # reemplaza el mes si se reprocesa
    hist = pd.concat([hist, panel], ignore_index=True)
else:
    hist = panel.copy()
# --- verificaciones contra el mes SBS anterior (unidades por departamento y cuadros repetidos) ---
prev_sbs = hist[(hist['fecha'] < FECHA) & (hist['fuente'] == 'sbs')]
ref = (prev_sbs[prev_sbs['fecha'] == prev_sbs['fecha'].max()]
       .groupby(['tipo', 'departamento'], observed=True)[['cred_total', 'dep_total']].sum()) if len(prev_sbs) else None
for t, g in panel.groupby('tipo', observed=True):
    if ref is not None and t in ref.index.get_level_values(0):
        r = ref.loc[t]
        act = g.groupby('departamento')[['cred_total', 'dep_total']].sum()
        razon = (act['cred_total'] / r['cred_total'].reindex(act.index)).replace([np.inf, -np.inf], np.nan).dropna()
        razon = razon[r['cred_total'].reindex(razon.index) > 1000]            # solo departamentos con saldo relevante
        en_millones = razon[(razon > 1/2000) & (razon < 1/500)].index
        if len(en_millones):
            panel.loc[(panel['tipo'] == t) & panel['departamento'].isin(en_millones), COLS_VALOR] *= 1000
            todo = len(en_millones) == len(razon)
            ADVERTENCIAS.append(f'{t}: el archivo {"estaba" if todo else "mezclaba unidades y en parte estaba"} en millones de soles '
                                f'({len(en_millones)} de {len(razon)} departamentos); se multiplicó por 1 000 donde correspondía. '
                                + ('' if todo else 'Conviene volver a descargar el cuadro de la SBS.'))
        elif np.isclose(g['cred_total'].sum(), r['cred_total'].sum()) and np.isclose(g['dep_total'].sum(), r['dep_total'].sum()):
            ADVERTENCIAS.append(f'{t}: cifras idénticas al mes anterior; verificar que la SBS haya actualizado el cuadro.')
    elif g['cred_total'].sum() < 1e5:   # sin referencia: ningún tipo tiene menos de S/ 100 mill. de créditos a nivel nacional
        panel.loc[panel['tipo'] == t, COLS_VALOR] *= 1000
        ADVERTENCIAS.append(f'{t}: el archivo parece estar en millones de soles (se multiplicó por 1 000).')
hist = pd.concat([hist[hist['fecha'] != FECHA], panel], ignore_index=True)
hist.sort_values(['fecha', 'tipo', 'empresa']).to_csv(HIST, index=False)
meses_hist = sorted(hist['fecha'].unique())
print("Meses acumulados en el histórico:", [pd.Timestamp(m).strftime('%Y-%m') for m in meses_hist])
for a in ADVERTENCIAS:
    print('AVISO:', a)

In [ ]:
# 4b. Libro BANCA_MULTIPLE_SBS (acumulado) y siembra del histórico con sus hojas antiguas
import shutil
from openpyxl import load_workbook, Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

DEPARTAMENTOS = {'Loreto': 'LORETO', 'San Martin': 'SAN MARTÍN', 'Ucayali': 'UCAYALI'}   # clave = nombre SBS
MASTER = os.path.join(BASE_SECTOR, 'BANCA_MULTIPLE_SBS.xlsx')      # libro acumulado (una hoja por mes)
NOMBRE_HOJA = f'{MESES_ES[FECHA.month-1].capitalize()}-{FECHA.year}'
MES_ABREV = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Set', 'Oct', 'Nov', 'Dic'][FECHA.month-1]

# primera vez: se toma como base el libro histórico colocado en input (cualquier *SBS*.xlsx)
if not os.path.exists(MASTER):
    semilla = glob.glob(os.path.join(INPUT_DIR, '*SBS*.xlsx'))
    if semilla:
        shutil.copy(semilla[0], MASTER); print("Libro base copiado a", MASTER)
    else:
        Workbook().save(MASTER); print("Libro nuevo creado en", MASTER)

# --- normalización de nombres de entidad (hojas antiguas -> nombres SBS) ---
ALIAS = {
    'BBVA CONTINENTAL': 'BBVA', 'B B V A BANCO CONTIN': 'BBVA', 'CREDITO': 'BANCO DE CREDITO',
    'PICHINCHA': 'BANCO PICHINCHA', 'AZTECA DEL PERU': 'ALFIN BANCO', 'BANCO AZTECA': 'ALFIN BANCO',
    'BANCO DEL COMERCIO': 'BANCOM2', 'FINANCIERA EFECTIVA': 'BANCO EFECTIVA',
    'FINANCIERA SANTANDER CONSUMER': 'FINANCIERA SANTANDER CONSUMER', 'INFINANCEXP': 'INFINANCE XP',
    'CRAC NSTRA GENTE': 'CRAC NUESTRA GENTE', 'CRAC SIPAN': 'CRAC SIPAN',
    'EDPYME ALTERNATIVA': 'EMP.CRED.ALTERNATIVA', 'EDPYME PROGRESO': 'EMP.CRED.PROGRESO',
    'EDPYME RAIZ': 'EMP.CRED.RAIZ', 'EDPYME SOLIDARIDAD': 'EMP.CRED.SOLIDARIDAD',
    'EDPYME ACCESO CREDITICIO': 'EMP.CRED.ACCESO CREDITICIO', 'EDPYME INVERSIONES LA CRUZ': 'EMP.CRED.INVERSIONES LA CRUZ',
    'BANCO DE LA NACION': 'NACION',
}
TIPO_LEGADO = {  # tipo (clasificación SBS actual) de entidades que ya no figuran con ese nombre
    'BANCO PICHINCHA': 'Banca Múltiple', 'BANCO EFECTIVA': 'Banca Múltiple',
    'FINANCIERA SANTANDER CONSUMER': 'Empresas Financieras', 'COMPARTAMOS FINANCIERA': 'Empresas Financieras',
    'FINANCIERA OH': 'Empresas Financieras', 'MITSUI AUTO FINANCE': 'Empresas Financieras', 'INFINANCE XP': 'Empresas Financieras',
    'CRAC NUESTRA GENTE': 'Cajas Rurales', 'CRAC SIPAN': 'Cajas Rurales', 'CRAC CREDINKA': 'Cajas Rurales',
    'CRAC SEÑOR DE LUREN': 'Cajas Rurales', 'GMG SERVICIOS PERU': 'Empresas de Crédito',
}

def norm_entidad(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().upper()
    s = s.replace('C.M.A.C', 'CMAC').replace('C.R.A.C', 'CRAC').replace('.', ' ')
    s = re.sub(r'\bS\s*A\b', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return ALIAS.get(s, s)

TIPO_POR_ENTIDAD = {e: t for t, e in panel[['tipo', 'empresa']].drop_duplicates().itertuples(index=False)}
TIPO_POR_ENTIDAD.update({k: v for k, v in TIPO_LEGADO.items() if k not in TIPO_POR_ENTIDAD})
def tipo_de(e):
    if e in TIPO_POR_ENTIDAD: return TIPO_POR_ENTIDAD[e]
    if e.startswith('CMAC') or e.startswith('CMCP'): return 'Cajas Municipales'
    if e.startswith('CRAC'): return 'Cajas Rurales'
    if e.startswith('EMP.CRED') or e.startswith('EDPYME'): return 'Empresas de Crédito'
    if 'FINANC' in e: return 'Empresas Financieras'
    return 'Banca Múltiple'

# --- lector de hojas antiguas del libro (para sembrar el histórico y comparar oficinas) ---
MES_NUM = {m: i + 1 for i, m in enumerate(MESES_ES)}
def fecha_de_hoja(nombre):
    m = re.match(r'([A-Za-zñ]+)\s*-\s*(\d{4})', nombre.strip())
    if not m or m.group(1).lower() not in MES_NUM: return None
    return pd.Timestamp(int(m.group(2)), MES_NUM[m.group(1).lower()], 1) + pd.offsets.MonthEnd(0)

def leer_hoja_legado(ws):
    rows = []
    dpto = None
    for r in ws.iter_rows(min_row=1, max_row=ws.max_row, max_col=18, values_only=True):
        a = str(r[0]).strip() if r[0] is not None else ''
        if ':' in a and 'Oficina' in a:
            dpto = a.split(':')[0].strip().title()
            dpto = {'San Martín': 'San Martin', 'San Martin': 'San Martin'}.get(dpto, dpto)
            continue
        if not a or re.match(r'(?i)^(total|banca m|empresas fin|entidades micro|iquitos|empresa$|al \d|\(en)', a):
            continue
        vals = list(r[5:18])
        if all(v is None for v in vals) and a.upper() != 'AGROBANCO':
            continue                                   # oficina sin datos (cerrada o no reportada)
        if r[4] is None and a.upper() != 'AGROBANCO':
            continue
        v = [float(x) if isinstance(x, (int, float)) else 0.0 for x in vals]
        e = norm_entidad(a)
        rows.append([e, dpto, r[2], r[3], r[4]] + v)
    df = pd.DataFrame(rows, columns=['empresa', 'departamento', 'provincia', 'distrito', 'cod_oficina',
                                     'dep_vista_mn', 'dep_vista_me', 'dep_vista_total', 'dep_ahorro_mn', 'dep_ahorro_me',
                                     'dep_ahorro_total', 'dep_plazo_mn', 'dep_plazo_me', 'dep_plazo_total', 'dep_total',
                                     'cred_mn', 'cred_me', 'cred_total'])
    df.insert(0, 'tipo', df['empresa'].map(tipo_de)); df['nivel'] = 'oficina'
    return df

wb = load_workbook(MASTER)

# sembrar el histórico con las hojas antiguas del libro (solo meses que no estén ya)
nuevos = []
for ws in wb.worksheets:
    f = fecha_de_hoja(ws.title)
    if f is None or f in set(hist['fecha']): continue
    h = leer_hoja_legado(ws)
    if len(h):
        h.insert(0, 'fecha', f); h['fuente'] = 'libro'; nuevos.append(h[COLS_ID + COLS_VALOR + ['fuente']])
if nuevos:
    hist = pd.concat([hist] + nuevos, ignore_index=True)
    hist['fecha'] = pd.to_datetime(hist['fecha'])
    hist.sort_values(['fecha', 'tipo', 'empresa']).to_csv(HIST, index=False)
    meses_hist = sorted(hist['fecha'].unique())
    print("Histórico sembrado desde las hojas del libro:", [pd.Timestamp(h0['fecha'].iloc[0]).strftime('%Y-%m') for h0 in nuevos])


# provincia para los cuadros sin esa columna (diseño C): se toma del histórico por oficina
clave = lambda x: unicodedata.normalize('NFKD', str(x)).encode('ascii', 'ignore').decode().lower().strip()
ref_prov = hist[(hist['nivel'] == 'oficina') & hist['provincia'].notna()].copy()
ref_prov['orden'] = (ref_prov['fuente'] != 'sbs').astype(int)            # primero nombres de archivos SBS
ref_prov['k'] = list(zip(ref_prov['departamento'].map(clave), ref_prov['distrito'].map(clave)))
mapa_prov = ref_prov.sort_values(['orden', 'fecha'], ascending=[True, False]).drop_duplicates('k').set_index('k')['provincia']
sin_prov = panel['provincia'].isna()
panel.loc[sin_prov, 'provincia'] = [mapa_prov.get((clave(d), clave(x)), 'Provincia no identificada')
                                    for d, x in zip(panel.loc[sin_prov, 'departamento'], panel.loc[sin_prov, 'distrito'])]
hist.loc[hist['fecha'] == FECHA, 'provincia'] = panel['provincia'].values
hist.sort_values(['fecha', 'tipo', 'empresa']).to_csv(HIST, index=False)

In [ ]:
# 5. Libro BANCA_MULTIPLE_SBS: hoja del mes con el cuadro por oficina (Loreto, San Martín, Ucayali)
# --- construcción de la hoja del mes ---
F_TIT = PatternFill('solid', fgColor='00B050'); F_GRP = PatternFill('solid', fgColor='92D050'); F_HDR = PatternFill('solid', fgColor='D9D9D9')
B = Font(bold=True); NUMFMT = '#,##0'
fino = Side(style='thin', color='999999'); BORDE = Border(top=fino, bottom=fino, left=fino, right=fino)
VAL_COLS = COLS_VALOR                      # 13 columnas de valores, en el orden F..R
NO_APLICA = {'Cajas Municipales': {'dep_vista_mn', 'dep_vista_me', 'dep_vista_total'},
             'Cajas Rurales': {'dep_vista_mn', 'dep_vista_me', 'dep_vista_total'},
             'Empresas de Crédito': set(c for c in COLS_VALOR if c.startswith('dep_'))}

for nombre in [n for n in wb.sheetnames if n.strip().lower() == NOMBRE_HOJA.lower()]:
    del wb[nombre]                                   # reemplaza la hoja del mes si ya existía
if 'Sheet' in wb.sheetnames and wb['Sheet'].max_row == 1:
    del wb['Sheet']
ws = wb.create_sheet(NOMBRE_HOJA)
for col, w in zip('ABCDE', [38, 14, 24, 24, 9]): ws.column_dimensions[col].width = w
for c in range(6, 19): ws.column_dimensions[get_column_letter(c)].width = 13

def fila_valores(r, vals, bold=False, fill=None):
    for j, v in enumerate(vals):
        cell = ws.cell(r, 6 + j, v); cell.number_format = NUMFMT
        if bold: cell.font = B
        if fill: cell.fill = fill
def fila_formula(r, texto, refs, fill=None, bold=True):
    """refs: lista de filas (se suman) o tupla (ini, fin) para SUM de rango."""
    ws.cell(r, 1, texto).font = B
    for j in range(13):
        L = get_column_letter(6 + j)
        if isinstance(refs, tuple): f = f'=SUM({L}{refs[0]}:{L}{refs[1]})'
        else: f = '=' + '+'.join(f'{L}{k}' for k in refs) if refs else '=0'
        cell = ws.cell(r, 6 + j, f); cell.number_format = NUMFMT; cell.font = B
        if fill: cell.fill = fill
    if fill:
        for c in range(1, 6): ws.cell(r, c).fill = fill

r = 1
for dkey, dname in DEPARTAMENTOS.items():
    d = panel[panel['departamento'].str.lower() == dkey.lower()].copy()
    d['prov'] = d['provincia']
    ws.cell(r, 1, f'{dname}: Depósitos y Créditos por Oficina y Empresa').font = Font(bold=True, color='FFFFFF')
    for c in range(1, 19): ws.cell(r, c).fill = F_TIT
    ws.cell(r + 1, 1, f'Al {FECHA.day} de {MESES_ES[FECHA.month-1].capitalize()} del {FECHA.year}').font = B
    ws.cell(r + 2, 1, '(En miles de soles)')
    if not TODO_POR_OFICINA:
        ws.cell(r + 2, 6, 'Sin detalle por entidad y oficina: ' + ', '.join(NIVEL_OFICINA.index[~NIVEL_OFICINA])).font = Font(italic=True, color='C00000')
    hdr1 = ['Empresa', 'Ubicación', None, None, 'Código Oficina', 'Depósitos a la Vista', None, None, 'Depósitos de Ahorro',
            None, None, 'Depósitos a Plazo', None, None, 'Total Depósitos', 'Créditos Directos', None, 'Total Créditos']
    hdr2 = [None, 'Departamento', 'Provincia', 'Distrito', None, 'M.N', 'M.E.', 'Total', 'M.N', 'M.E.', 'Total', 'M.N', 'M.E.', 'Total', None, 'M.N', 'M.E.', None]
    for c, (a, b) in enumerate(zip(hdr1, hdr2), start=1):
        for rr, v in ((r + 3, a), (r + 4, b)):
            cell = ws.cell(rr, c, v); cell.font = B; cell.fill = F_HDR; cell.border = BORDE
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    for rng in ['A{0}:A{1}', 'B{0}:D{0}', 'E{0}:E{1}', 'F{0}:H{0}', 'I{0}:K{0}', 'L{0}:N{0}', 'O{0}:O{1}', 'P{0}:Q{0}', 'R{0}:R{1}']:
        ws.merge_cells(rng.format(r + 3, r + 4))
    r += 5

    ORDEN_PREF = ['BBVA', 'BANCO DE CREDITO', 'INTERBANK', 'SCOTIABANK PERU', 'MIBANCO', 'BANBIF', 'BANCO FALABELLA',
                  'BANCO RIPLEY', 'COMPARTAMOS BANCO', 'CMAC MAYNAS', 'CMAC PIURA', 'CMAC HUANCAYO']
    orden = lambda e: (ORDEN_PREF.index(e) if e in ORDEN_PREF else 99, e)
    filas_grupo = {}          # tipo -> fila del total de grupo
    for tipo in ORDEN_TIPOS:
        dt = d[d['tipo'] == tipo]
        if tipo == 'Entidades Estatales':
            continue          # se tratan aparte (Banco de la Nación y Agrobanco)
        filas_ent = []
        if len(dt) and not NIVEL_OFICINA[tipo]:        # cuadro SBS agregado: una fila por distrito, sin entidad
            dt = dt.assign(empresa=f'{tipo.upper()} (cuadro SBS sin detalle por entidad)')
        for emp in sorted(dt['empresa'].unique(), key=orden):
            de = dt[dt['empresa'] == emp].sort_values(['provincia', 'distrito', 'cod_oficina'])
            ini = r
            for _, o in de.iterrows():
                ws.cell(r, 1, emp); ws.cell(r, 2, dname.title()); ws.cell(r, 3, o['prov']); ws.cell(r, 4, o['distrito'])
                ws.cell(r, 5, int(o['cod_oficina']) if pd.notna(o['cod_oficina']) else None)
                fila_valores(r, [None if c in NO_APLICA.get(tipo, set()) else o[c] for c in VAL_COLS]); r += 1
            if not NIVEL_OFICINA[tipo]:
                filas_ent.extend(range(ini, r))            # agregado: el total de grupo suma los distritos
            elif len(de) > 1:
                fila_formula(r, f'TOTAL {emp}', (ini, r - 1)); filas_ent.append(r); r += 1
            else:
                filas_ent.append(ini)
        etiqueta = {'Banca Múltiple': 'BANCA MÚLTIPLE', 'Empresas Financieras': 'EMPRESAS FINANCIERAS',
                    'Cajas Municipales': 'TOTAL CAJAS MUNICIPALES', 'Cajas Rurales': 'TOTAL CAJAS RURALES',
                    'Empresas de Crédito': 'TOTAL EMPRESAS DE CRÉDITO (EDPYME)'}[tipo]
        fila_formula(r, etiqueta, filas_ent, fill=F_GRP); filas_grupo[tipo] = r; r += 1
    fila_formula(r, 'ENTIDADES MICROFINANCIERAS NO BANCARIAS',
                 [filas_grupo[t] for t in ('Cajas Municipales', 'Cajas Rurales', 'Empresas de Crédito')], fill=F_GRP)
    fila_emnb = r; r += 1

    # Banco de la Nación y Agrobanco
    filas_estatales = []
    for emp, etiqueta in (('NACION', 'TOTAL BANCO DE LA NACIÓN'), ('AGROBANCO', 'AGROBANCO')):
        de = d[(d['tipo'] == 'Entidades Estatales') & (d['empresa'] == emp)].sort_values(['provincia', 'distrito', 'cod_oficina'])
        ini = r
        for _, o in de.iterrows():
            ws.cell(r, 1, 'Banco de la Nación' if emp == 'NACION' else 'Agrobanco'); ws.cell(r, 2, dname.title())
            ws.cell(r, 3, o['prov']); ws.cell(r, 4, o['distrito'])
            ws.cell(r, 5, int(o['cod_oficina']) if pd.notna(o['cod_oficina']) else None)
            fila_valores(r, [o[c] for c in VAL_COLS]); r += 1
        fila_formula(r, etiqueta, (ini, r - 1) if r > ini else [], fill=F_GRP); filas_estatales.append(r); r += 1

    fila_formula(r, f'TOTAL SISTEMA FINANCIERO {dname}',
                 [filas_grupo['Banca Múltiple'], filas_grupo['Empresas Financieras'], fila_emnb] + filas_estatales, fill=F_TIT)
    for c in range(1, 19): ws.cell(r, c).font = Font(bold=True, color='FFFFFF')
    r += 3

ws.freeze_panes = 'F6'
wb.save(MASTER)
salida_libro = os.path.join(OUTPUT_DIR, f'BANCA_MULTIPLE_SBS-{MES_ABREV}{FECHA.strftime("%y")}.xlsx')
shutil.copy(MASTER, salida_libro)
print(f"Hoja '{NOMBRE_HOJA}' agregada. Libro acumulado: {MASTER}\nCopia del mes: {salida_libro}")

In [ ]:
# 6. REPORTE de aperturas y cierres de oficinas respecto al mes anterior
GRUPO = {'Banca Múltiple': 'BANCA MÚLTIPLE', 'Empresas Financieras': 'EMPRESAS FINANCIERAS',
         'Cajas Municipales': 'CAJAS MUNICIPALES', 'Cajas Rurales': 'CAJAS RURALES',
         'Empresas de Crédito': 'EMPRESAS DE CRÉDITO (EDPYME)', 'Entidades Estatales': 'ENTIDADES ESTATALES'}
NOMBRE_LARGO = {'NACION': 'Banco de la Nación', 'BANCO DE CREDITO': 'Banco de Crédito'}

def oficinas(df):
    d = df[df['departamento'].isin(DEPARTAMENTOS)].dropna(subset=['cod_oficina'])
    return d.assign(cod_oficina=d['cod_oficina'].astype(int)).set_index(['departamento', 'empresa', 'cod_oficina'])[['tipo', 'provincia', 'distrito']]

prev = [m for m in meses_hist if m < FECHA]
lineas = [f'REPORTE DE {MESES_ES[FECHA.month-1].upper()} {FECHA.year}:', '']
if not prev:
    lineas.append('- No hay mes anterior en el histórico para comparar.')
else:
    f_prev = pd.Timestamp(max(prev))
    h_prev = hist[hist['fecha'] == f_prev]
    sin_codigo = set(h_prev.loc[h_prev['cod_oficina'].isna() & h_prev['departamento'].isin(DEPARTAMENTOS), 'empresa'].dropna())
    niv_prev = h_prev.groupby('tipo')['nivel'].agg(lambda x: (x == 'oficina').all())
    comparables = [t for t in ORDEN_TIPOS if NIVEL_OFICINA.get(t, False) and niv_prev.get(t, False)]
    no_comp = [t for t in ORDEN_TIPOS if t not in comparables]
    act, ant = oficinas(panel[panel['tipo'].isin(comparables)]), oficinas(h_prev[h_prev['tipo'].isin(comparables)])
    act = act[~act.index.get_level_values('empresa').isin(sin_codigo)]
    ant = ant[~ant.index.get_level_values('empresa').isin(sin_codigo)]
    nuevas = act.loc[~act.index.isin(ant.index)].reset_index()
    cerradas = ant.loc[~ant.index.isin(act.index)].reset_index()
    vistas_antes = oficinas(hist[hist['fecha'] < f_prev]).index          # para distinguir apertura de reaparición
    lineas.append(f'(comparación con {MESES_ES[f_prev.month-1]} {f_prev.year}'
                  + ('; mes anterior tomado de la hoja del libro, no del archivo SBS)' if (h_prev['fuente'] == 'libro').any() else ')'))
    if no_comp:
        lineas.append(f"- Sin comparación de oficinas para {', '.join(no_comp)}: falta el detalle por oficina en este mes o en el anterior.")
    if sin_codigo:
        lineas.append(f"- No se comparan oficinas de {', '.join(sorted(sin_codigo))}: el mes anterior no registra códigos de oficina.")
    for _, o in nuevas.sort_values(['departamento', 'tipo', 'empresa']).iterrows():
        verbo = 'reaparece' if (o['departamento'], o['empresa'], o['cod_oficina']) in vistas_antes else 'se abrió'
        lineas.append(f"- En {DEPARTAMENTOS[o['departamento']].title()}, en {GRUPO[str(o['tipo'])]}, {verbo} una oficina de "
                      f"{NOMBRE_LARGO.get(o['empresa'], o['empresa'].title())}, código de oficina {o['cod_oficina']} "
                      f"({o['distrito']}, {o['provincia']}).")
    for _, o in cerradas.sort_values(['departamento', 'tipo', 'empresa']).iterrows():
        lineas.append(f"- En {DEPARTAMENTOS[o['departamento']].title()}, en {GRUPO[str(o['tipo'])]}, se cerró una oficina de "
                      f"{NOMBRE_LARGO.get(o['empresa'], o['empresa'].title())}, código de oficina {o['cod_oficina']} "
                      f"({o['distrito']}, {o['provincia']}).")
    if len(nuevas) + len(cerradas) == 0:
        lineas.append('- Sin aperturas ni cierres de oficinas.')
    lineas += ['', f'Oficinas activas por departamento ({MESES_ES[f_prev.month-1]} -> {MESES_ES[FECHA.month-1]}):']
    for dk, dn in DEPARTAMENTOS.items():
        na, nb = (ant.reset_index()['departamento'] == dk).sum(), (act.reset_index()['departamento'] == dk).sum()
        lineas.append(f'  {dn.title():12s} {na:4d} -> {nb:4d}')
lineas += ['', 'ADVERTENCIAS DE LOS INPUTS:'] + ([f'- {a}' for a in ADVERTENCIAS] or ['- Ninguna.'])
reporte = '\n'.join(lineas)
with open(os.path.join(OUTPUT_DIR, f'REPORTE_{TAG}.txt'), 'w', encoding='utf-8') as fh:
    fh.write(reporte)
print(reporte)

In [ ]:
# 7. Análisis por departamento: cuadros, variaciones, Excel y gráficos (una corrida para los tres)
PAL = {'Banca Múltiple': '#2a78d6', 'Empresas Financieras': '#eb6834', 'Cajas Municipales': '#1baf7a',
       'Cajas Rurales': '#eda100', 'Empresas de Crédito': '#e87ba4', 'Entidades Estatales': '#4a3aa7'}
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.25, 'axes.axisbelow': True, 'figure.dpi': 120})
fmt_mill = mtick.FuncFormatter(lambda x, _: f'{x/1e3:,.0f}')
titulo_mes = f'{MESES_ES[FECHA.month-1]} {FECHA.year}'

def analizar_departamento(dkey, dname):
    """Genera Financiero_<Depto>_<AAAAMM>.xlsx y los gráficos g1..g5 del departamento."""
    # --- cuadros del departamento ---
    DEPARTAMENTO = dname.title() if dname != 'SAN MARTÍN' else 'San Martín'
    DEP_TAG = unicodedata.normalize('NFKD', dkey).encode('ascii', 'ignore').decode().replace(' ', '')
    dep = panel[panel['departamento'].str.lower() == dkey.lower()].copy()
    nac = panel

    def agrega(df, por):
        g = df.groupby(por, observed=True).agg(
            oficinas=('cod_oficina', 'count'),
            cred_mn=('cred_mn', 'sum'), cred_me=('cred_me', 'sum'), cred_total=('cred_total', 'sum'),
            dep_vista=('dep_vista_total', 'sum'), dep_ahorro=('dep_ahorro_total', 'sum'),
            dep_plazo=('dep_plazo_total', 'sum'),
            dep_mn=('dep_vista_mn', 'sum'), dep_me=('dep_vista_me', 'sum'), dep_total=('dep_total', 'sum'))
        # MN/ME de depósitos = suma de los tres productos
        g['dep_mn'] = df.groupby(por, observed=True)[['dep_vista_mn', 'dep_ahorro_mn', 'dep_plazo_mn']].sum().sum(axis=1)
        g['dep_me'] = df.groupby(por, observed=True)[['dep_vista_me', 'dep_ahorro_me', 'dep_plazo_me']].sum().sum(axis=1)
        g['part_cred_%'] = 100 * g['cred_total'] / g['cred_total'].sum()
        g['part_dep_%']  = 100 * g['dep_total']  / g['dep_total'].sum()
        g['dolariz_cred_%'] = 100 * g['cred_me'] / g['cred_total'].replace(0, np.nan)
        g['dolariz_dep_%']  = 100 * g['dep_me']  / g['dep_total'].replace(0, np.nan)
        g['cred/dep'] = g['cred_total'] / g['dep_total'].replace(0, np.nan)
        return g

    por_tipo      = agrega(dep, 'tipo')
    por_empresa   = agrega(dep, ['tipo', 'empresa']).sort_values('cred_total', ascending=False)
    por_provincia = agrega(dep, 'provincia').sort_values('cred_total', ascending=False)
    por_distrito  = agrega(dep, ['provincia', 'distrito']).sort_values('cred_total', ascending=False)
    por_tipo_prov = dep.pivot_table(index='provincia', columns='tipo', values='cred_total',
                                    aggfunc='sum', observed=True, fill_value=0)

    # ranking nacional por departamento (para situar al departamento)
    nac_dpto = agrega(nac, 'departamento').sort_values('cred_total', ascending=False)
    nac_dpto['rank_cred'] = nac_dpto['cred_total'].rank(ascending=False).astype(int)
    nac_dpto['rank_dep']  = nac_dpto['dep_total'].rank(ascending=False).astype(int)

    # resumen de una página
    tot_d, tot_n = agrega(dep, 'fecha').iloc[0], agrega(nac, 'fecha').iloc[0]
    fila_dep = nac_dpto.loc[nac_dpto.index.str.lower() == dkey.lower()].iloc[0]
    resumen = pd.DataFrame({
        'Indicador': ['Fecha de corte', 'Créditos directos (mill. S/)', 'Depósitos (mill. S/)',
                      'Oficinas', 'Participación en créditos nacionales (%)', 'Participación en depósitos nacionales (%)',
                      'Ranking departamental créditos', 'Ranking departamental depósitos',
                      'Dolarización de créditos (%)', 'Dolarización de depósitos (%)', 'Ratio créditos/depósitos',
                      'Nº entidades con oficina en el departamento'],
        'Valor': [FECHA.strftime('%d/%m/%Y'), tot_d['cred_total'] / 1e3, tot_d['dep_total'] / 1e3, int(tot_d['oficinas']),
                  100 * tot_d['cred_total'] / tot_n['cred_total'], 100 * tot_d['dep_total'] / tot_n['dep_total'],
                  int(fila_dep['rank_cred']), int(fila_dep['rank_dep']),
                  tot_d['dolariz_cred_%'], tot_d['dolariz_dep_%'], tot_d['cred/dep'], dep['empresa'].nunique()]
    })
    if not TODO_POR_OFICINA:
        resumen.loc[resumen['Indicador'].isin(['Oficinas', 'Nº entidades con oficina en el departamento']), 'Valor'] = 'n.d. (inputs sin detalle por oficina)'
    for a in ADVERTENCIAS:
        resumen.loc[len(resumen)] = ['Advertencia', a]
    print(resumen.to_string(index=False))
    print("\nPor tipo de entidad (miles de S/):")
    print(por_tipo[['oficinas', 'cred_total', 'part_cred_%', 'dep_total', 'part_dep_%', 'dolariz_cred_%']])

    # --- variaciones (si hay meses anteriores) ---
    variaciones = None
    if len(meses_hist) > 1:
        hd = hist[hist['departamento'].str.lower() == dkey.lower()]
        serie = hd.groupby(['fecha', 'tipo'], observed=True)[['cred_total', 'dep_total']].sum().unstack('tipo')
        serie_tot = hd.groupby('fecha')[['cred_total', 'dep_total']].sum()
        variaciones = pd.DataFrame({
            'cred_total': serie_tot['cred_total'], 'dep_total': serie_tot['dep_total'],
            'var_mensual_cred_%': 100 * serie_tot['cred_total'].pct_change(),
            'var_mensual_dep_%':  100 * serie_tot['dep_total'].pct_change(),
        })
        f12 = FECHA - pd.DateOffset(years=1)
        if f12 in serie_tot.index:
            variaciones.loc[FECHA, 'var_interanual_cred_%'] = 100 * (serie_tot.loc[FECHA, 'cred_total'] / serie_tot.loc[f12, 'cred_total'] - 1)
            variaciones.loc[FECHA, 'var_interanual_dep_%']  = 100 * (serie_tot.loc[FECHA, 'dep_total']  / serie_tot.loc[f12, 'dep_total']  - 1)
        print(variaciones.tail(13))
    else:
        print("Primer mes en el histórico: las variaciones se calcularán desde el próximo mes.")

    # --- exportar a Excel ---
    xlsx = os.path.join(OUTPUT_DIR, f'Financiero_{DEP_TAG}_{TAG}.xlsx')
    with pd.ExcelWriter(xlsx, engine='openpyxl') as w:
        resumen.to_excel(w, sheet_name='Resumen', index=False)
        por_tipo.to_excel(w, sheet_name='Por tipo')
        por_empresa.to_excel(w, sheet_name='Por empresa')
        por_provincia.to_excel(w, sheet_name='Por provincia')
        por_distrito.to_excel(w, sheet_name='Por distrito')
        por_tipo_prov.to_excel(w, sheet_name='Tipo x provincia')
        dep.drop(columns=['fecha', 'fuente']).to_excel(w, sheet_name='Oficinas', index=False)
        nac_dpto.to_excel(w, sheet_name='Ranking nacional')
        if variaciones is not None:
            variaciones.to_excel(w, sheet_name='Variaciones')
        if validacion is not None:
            validacion[validacion['departamento'].str.lower() == dkey.lower()].to_excel(w, sheet_name='Validación', index=False)
        for ws in w.book.worksheets:                      # ancho de columnas
            for col in ws.columns:
                ws.column_dimensions[col[0].column_letter].width = min(
                    45, max(10, max(len(str(c.value)) if c.value is not None else 0 for c in col) + 2))
    print("Guardado:", xlsx)

    # --- gráficos ---
    def guarda(fig, nombre):
        p = os.path.join(OUTPUT_DIR, f'{nombre}_{DEP_TAG}_{TAG}.png')
        fig.savefig(p, bbox_inches='tight'); plt.show(); plt.close(fig); print("Guardado:", p)

    # 8a. Créditos y depósitos por tipo de entidad (dos paneles, misma escala)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for ax, col, t in zip(axes, ['cred_total', 'dep_total'], ['Créditos directos', 'Depósitos']):
        s = por_tipo[col]
        ax.barh(s.index.astype(str), s.values, color=[PAL[i] for i in s.index], height=0.6)
        for i, v in enumerate(s.values):
            ax.text(v, i, f'  {v/1e3:,.0f}', va='center', fontsize=9)
        ax.set_title(f'{t} (mill. S/)'); ax.xaxis.set_major_formatter(fmt_mill); ax.invert_yaxis()
    fig.suptitle(f'{DEPARTAMENTO}: sistema financiero por tipo de entidad, {titulo_mes}', y=1.03)
    guarda(fig, 'g1_por_tipo')

    # 8b. Top 10 entidades por créditos y depósitos (solo con detalle por entidad)
    top = por_empresa.reset_index().head(10) if TODO_POR_OFICINA else pd.DataFrame()
    if len(top):
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        for ax, col, t in zip(axes, ['cred_total', 'dep_total'], ['Créditos directos', 'Depósitos']):
            d = top.sort_values(col, ascending=True)
            ax.barh(d['empresa'], d[col], color=[PAL[x] for x in d['tipo']], height=0.65)
            ax.set_title(f'{t} (mill. S/)'); ax.xaxis.set_major_formatter(fmt_mill)
        handles = [plt.Rectangle((0, 0), 1, 1, color=PAL[k]) for k in ORDEN_TIPOS if k in set(top['tipo'])]
        fig.legend(handles, [k for k in ORDEN_TIPOS if k in set(top['tipo'])], ncol=3, loc='lower center',
                   bbox_to_anchor=(0.5, -0.08), frameon=False)
        fig.suptitle(f'{DEPARTAMENTO}: principales entidades, {titulo_mes}', y=1.02)
        fig.tight_layout(); guarda(fig, 'g2_top_entidades')

    # 8c. Créditos por provincia, apilado por tipo de entidad
    d = por_tipo_prov.loc[por_provincia.index[::-1]]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    left = np.zeros(len(d))
    for t in ORDEN_TIPOS:
        if t in d.columns and d[t].sum() > 0:
            ax.barh(d.index, d[t], left=left, color=PAL[t], label=t, height=0.65); left += d[t].values
    ax.xaxis.set_major_formatter(fmt_mill); ax.set_xlabel('Créditos directos (mill. S/)')
    ax.legend(frameon=False, ncol=2, fontsize=8, loc='lower right')
    ax.set_title(f'{DEPARTAMENTO}: créditos directos por provincia y tipo de entidad, {titulo_mes}')
    guarda(fig, 'g3_provincias')

    # 8d. Ranking nacional por departamento (resalta el departamento)
    d = nac_dpto['cred_total'].sort_values()
    colores = ['#2a78d6' if i.lower() == dkey.lower() else '#c3c2b7' for i in d.index]
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.barh(d.index, d.values, color=colores, height=0.7)
    ax.set_xscale('log'); ax.set_xlabel('Créditos directos (mill. S/, escala log)')
    ax.xaxis.set_major_formatter(fmt_mill)
    ax.set_title(f'Créditos directos por departamento, {titulo_mes} (puesto {int(fila_dep["rank_cred"])} de {len(d)})')
    guarda(fig, 'g4_ranking_departamental')

    # 8e. Evolución (solo cuando hay histórico)
    if variaciones is not None and len(variaciones) > 1:
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(variaciones.index, variaciones['cred_total'], color='#2a78d6', lw=2, marker='o', ms=4, label='Créditos directos')
        ax.plot(variaciones.index, variaciones['dep_total'], color='#eb6834', lw=2, marker='o', ms=4, label='Depósitos')
        ax.yaxis.set_major_formatter(fmt_mill); ax.set_ylabel('mill. S/'); ax.legend(frameon=False)
        ax.set_title(f'{DEPARTAMENTO}: evolución de créditos y depósitos')
        guarda(fig, 'g5_evolucion')

panel.to_csv(os.path.join(OUTPUT_DIR, f'panel_oficinas_nacional_{TAG}.csv'), index=False)
for dkey, dname in DEPARTAMENTOS.items():
    print(f'\n{"="*30} {dname} {"="*30}')
    analizar_departamento(dkey, dname)

print("\nListo. Outputs en:", OUTPUT_DIR)
for n in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", n)